# Setup

In [ ]:
import pandas as pd
from experiments.modules.crows_runner import CrowSPairsRunnerWrapper

runner = CrowSPairsRunnerWrapper(
    # model_name_or_path="jhu-clsp/mmBERT-base",
    batched=True,
)

models_mbert = {
    "inlp": "INLPBertForMaskedLM", 
    "densray": "DensrayDebiasBertForMaskedLM", 
    "sentdebias": "SentenceDebiasBertForMaskedLM", 
    "cda": "CDABertForMaskedLM", 
    "dropout": "DropoutBertForMaskedLM"
}

models_mmbert = {
    "inlp": "INLPModernBertForMaskedLM", 
    "densray": "DensRayModernBertForMaskedLM", 
    "sentdebias": "SentenceDebiasModernBertForMaskedLM", 
    "cda": "CDAModernBertForMaskedLM", 
    "dropout": "DropoutModernBertForMaskedLM"
}

models = models_mmbert

eval_languages = ["ar_DZ", "ca_ES", "de_DE", "en_US", "es_AR", "fr_FR", "it_IT", "mt_MT", "zh_CN"]
eval_btype = ["gender", "race-color", "religion"]
debias_languages = ["ar_DZ", "ca_ES", "de_DE", "en_US", "es_AR", "fr_FR", "it_IT", "mt_MT", "zh_CN"]
debias_methods = ["cda", "dropout", "inlp", "densray", "sentdebias"]

# Base Score

In [ ]:
results = {}
for elang in eval_languages:
    btype_result = {}
    for btype in eval_btype:
        btype_result[btype] = runner.run_base(
            path_to_crows=f"data/crows_improved/crows_{elang}.csv",
            lang_eval=elang,
            bias_type=btype
        )[0]
    results[elang] = btype_result

print(results)

# Debiased Score

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

results_folger = "results_0908"
results={}
for method in debias_methods:
    print(f"Method: {method}", flush=True)
    dlang_result = {}
    for dlang in debias_languages:
        # print(f"DebLang: {dlang}:")
        elang_result = {}
        for elang in eval_languages:
            # print(f"EvalLang: {elang}:")
            btype_result = {}
            for btype in eval_btype:
                # print(f"BType: {btype}:")
                if method in ["densray", "densray_big"] and btype != "gender":
                    continue
                print(models[method])
                btype_result[btype] = runner.run_debias(
                    model_class=models[method],
                    path_to_crows=f"data/crows_improved/crows_{elang}.csv",
                    lang_debias=dlang,
                    lang_eval=elang,
                    bias_type=btype,
                    bias_direction=f"{results_folger}/{method}/{dlang}/{btype}.pt" if method in ["densray", "densray_big", "sentdebias", "sentdebias_big"] else None,
                    projection_matrix=f"{results_folger}/{method}/{dlang}/{btype}.pt" if method == "inlp" else None,
                    load_path=f"{results_folger}/{method}/{dlang}/{btype}/best_model" if method == "cda" else (f"{results_folger}/{method}/{dlang}/best_model" if method == "dropout" else None),
                )[0]
            elang_result[elang] = btype_result
        dlang_result[dlang] = elang_result
    results[method] = dlang_result

# Save results

In [ ]:
import json
import time
print(results)
print(dlang_result)
print(elang_result)
print(btype_result)
with open(f"data_{time.time()}.json", "w", encoding="utf-8") as file:
    json.dump(results, file)
